In [5]:
# ===== CÉLULA 1: IMPORTAÇÕES =====
import sys
import re
from pathlib import Path
from datetime import datetime

import pandas as pd
import pdfplumber

In [ ]:
# ===== CÉLULA 2: CAMINHOS E ARQUIVOS =====
def get_base_dir() -> Path:
    """Retorna o diretório base, funcionando em script, notebook e executável PyInstaller."""
    if getattr(sys, "frozen", False):
        # Empacotado: os PDFs ficam na pasta do .exe
        return Path(sys.executable).parent
    # Notebook/script: pasta do próprio arquivo (ou cwd como fallback)
    return Path(__file__).parent if "__file__" in globals() else Path.cwd()


BASE_DIR = get_base_dir()
PASTA_PLACAS = BASE_DIR / "Placas"

PADRAO_ARQUIVO = re.compile(
    r"notificacao(?P<tipo>Autuacao|Penalidade)_(?P<placa>[A-Z0-9]+)_(?P<numero>\d+)\.pdf$",
    re.IGNORECASE,
)


def encontrar_arquivos(pasta: Path = PASTA_PLACAS) -> list[dict]:
    """Varre Placas/<PLACA>/<Status>/*.pdf e retorna metadados de cada arquivo."""
    registros = []
    if not pasta.exists():
        print(f"Aviso: pasta não encontrada: {pasta}")
        return registros

    for pdf in pasta.rglob("*.pdf"):
        # ex.: ('BSC4C89', 'A_Vencer', 'arquivo.pdf')
        partes = pdf.relative_to(pasta).parts
        placa = partes[0] if len(partes) >= 2 else ""
        status = partes[1] if len(partes) >= 3 else "Sem_Status"

        m = PADRAO_ARQUIVO.search(pdf.name)
        tipo = m.group("tipo") if m else ""
        numero = m.group("numero") if m else ""

        registros.append({
            "placa": placa,
            "status": status,
            "tipo_notificacao": tipo,
            "numero": numero,
            "arquivo": pdf.name,
            "caminho": str(pdf),
        })
    return registros


ARQUIVOS = encontrar_arquivos()
print(f"{len(ARQUIVOS)} PDF(s) encontrados.")

19 PDF(s) encontrados.


In [ ]:
# ===== CÉLULA 3: LEITURA DOS PDFs =====
def ler_pdf(caminho: str) -> str:
    """Extrai o texto de um PDF. Retorna string vazia em caso de falha."""
    try:
        with pdfplumber.open(caminho) as pdf:
            return "\n".join((p.extract_text() or "") for p in pdf.pages)
    except Exception as e:
        print(f"Erro ao ler {caminho}: {e}")
        return ""


def normalizar(texto: str) -> str:
    """Colapsa quebras de linha e espaços múltiplos."""
    return re.sub(r"\s+", " ", texto).strip()


def extrair_campos(texto: str) -> dict:
    """Extrai campos das notificações (layout PRF/prefeituras)."""
    t = normalizar(texto)
    padroes = {
        "ait":                r"AIT\)\s*([A-Z0-9]{9,12})",
        "placa":              r"\bPLACA\s+([A-Z]{3}\d[A-Z0-9]\d{2})\b",
        "data_notificacao":   r"DATA DA NOTIFICAÇÃO DA (?:AUTUAÇÃO|PENALIDADE)\s*(\d{2}/\d{2}/\d{4})",
        "data_vencimento":    r"DATA DE VENCIMENTO DA PENALIDADE.*?(\d{2}/\d{2}/\d{4})",
        "data_limite_defesa": r"INTERPOSIÇÃO DE DEFESA PRÉVIA\s*(\d{2}/\d{2}/\d{4})",
        "data_limite_recurso": r"INTERPOSIÇÃO DE RECURSO\s*(\d{2}/\d{2}/\d{4})",
        "valor":              r"VALOR DA MULTA\s*R\$\s*([\d.,]+)",
        "codigo_infracao":    r"CÓDIGO DA INFRAÇÃO\s*(\d{4})",
        "descricao_infracao": r"DESCRIÇÃO DA INFRAÇÃO\s*(.+?)\s*(?:MEDIÇÃO REALIZADA|NÚMERO RENAINF|$)",
        "municipio":          r"NOME DO MUNICÍPIO\s*(.+?)\s*(?:UF\b|CÓDIGO DA INFRAÇÃO)",
        "proprietario":       r"NOME DO PROPRIETÁRIO\s*(.+?)\s*(?:CNH|CPF)",
        "marca_modelo":       r"MARCA/MODELO/VERSÃO\s*(.+?)\s*(?:NOME\b|LOCAL DA INFRAÇÃO)",
    }
    campos = {}
    for chave, padrao in padroes.items():
        m = re.search(padrao, t, re.IGNORECASE)
        campos[chave] = m.group(1).strip() if m else None

    # Órgão autuador = linha imediatamente anterior ao rótulo do AIT (texto original, sem normalizar)
    m = re.search(r"([^\n]+)\n[^\n]*IDENTIFICAÇÃO DO AUTO DE INFRAÇÃO", texto)
    campos["orgao_autuador"] = m.group(1).strip() if m else None
    return campos


for reg in ARQUIVOS:
    texto = ler_pdf(reg["caminho"])
    reg["texto"] = texto
    reg.update(extrair_campos(texto))

print(f"{sum(1 for r in ARQUIVOS if r.get('valor'))} de {len(ARQUIVOS)} PDFs com valor extraído.")

0 de 19 PDFs com valor extraído.


In [8]:
# ===== CÉLULA 4: DATAFRAMES =====
COLUNAS = ["placa", "status", "tipo_notificacao", "ait", "numero",
           "data_notificacao", "data_vencimento", "data_limite_defesa", "data_limite_recurso",
           "valor", "codigo_infracao", "descricao_infracao", "municipio",
           "orgao_autuador", "proprietario", "marca_modelo", "arquivo"]

df_multas = pd.DataFrame(ARQUIVOS)
df_multas = df_multas.reindex(columns=COLUNAS + ["caminho", "texto"])
df_multas["valor_num"] = (
    df_multas["valor"]
    .astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

df_multas.head()

,placa,status,tipo_notificacao,ait,numero,data_notificacao,data_vencimento,data_limite_defesa,data_limite_recurso,valor,codigo_infracao,descricao_infracao,municipio,orgao_autuador,proprietario,marca_modelo,arquivo,caminho,texto,valor_num
0,None,a_vencer,,MAIRIPORA,,06/09/2026,NaN,26/10/2026,NaN,None,None,TRANSITAR EM VELOCIDADE SUPERIOR A MAXIMA PERM...,UF IDENTIFICAÇÃO DO AUTO DE INFRAÇÃO (Número d...,NOME DO MUNICÍPIO UF,REGISTRO FOTOGRÁFICO JOANOPOLIS AGROPECUARIA L...,FORD/CARGO 1415 NÚMERO DO EQUIPAMENTO OU INSTR...,notificacaoAutuacao_BSC4C89_000100J01062408274...,a:\Multas\Placas\BSC4C89\a_vencer\notificacaoA...,NOTIFICAÇÃO DE AUTUAÇÃO\nIDENTIFICAÇÃO DA AUTU...,NaN
1,None,a_vencer,,NaN,,26/08/2026,NaN,02/10/2026,NaN,None,None,"MULTA, POR NAO IDENTIFICACAO DO CONDUTOR INFRA...",UF IDENTIFICAÇÃO DO AUTO DE INFRAÇÃO (Número d...,NOME DO MUNICÍPIO UF,REGISTRO FOTOGRÁFICO JOANOPOLIS AGROPECUARIA L...,FORD/CARGO 1415 NÚMERO DO EQUIPAMENTO OU INSTR...,notificacaoAutuacao_BSC4C89_1262001AA970275550...,a:\Multas\Placas\BSC4C89\a_vencer\notificacaoA...,NOTIFICAÇÃO DE AUTUAÇÃO\nIDENTIFICAÇÃO DA AUTU...,NaN
2,None,a_vencer,,MAIRIPORA,,15/09/2026,NaN,29/10/2026,NaN,None,None,TRANSITAR EM LOCAL/HORARIO NAO PERMITIDO PELA ...,UF IDENTIFICAÇÃO DO AUTO DE INFRAÇÃO (Número d...,NOME DO MUNICÍPIO UF,REGISTRO FOTOGRÁFICO JOANOPOLIS AGROPECUARIA L...,FORD/CARGO 1415 NÚMERO DO EQUIPAMENTO OU INSTR...,notificacaoAutuacao_BSC4C89_266710R48059854657...,a:\Multas\Placas\BSC4C89\a_vencer\notificacaoA...,NOTIFICAÇÃO DE AUTUAÇÃO\nIDENTIFICAÇÃO DA AUTU...,NaN
3,None,a_vencer,,MAIRIPORA,,NaN,09/12/2026,NaN,NaN,None,None,DATA DE VENCIMENTO DA PENALIDADE (DATA LIMITE ...,UF IDENTIFICAÇÃO DO AUTO DE INFRAÇÃO (NÚMERO D...,NOME DO MUNICÍPIO UF,REGISTRO FOTOGRÁFICO JOANOPOLIS AGROPECUARIA L...,FORD/CARGO 1415 NÚMERO DO EQUIPAMENTO OU INSTR...,notificacaoPenalidade_BSC4C89_000100J007389472...,a:\Multas\Placas\BSC4C89\a_vencer\notificacaoP...,NOTIFICAÇÃO DE PENALIDADE\nIDENTIFICAÇÃO DA PE...,NaN
4,None,a_vencer,,NaN,,NaN,13/10/2026,NaN,NaN,None,None,DATA DE VENCIMENTO DA PENALIDADE (DATA LIMITE ...,UF IDENTIFICAÇÃO DO AUTO DE INFRAÇÃO (NÚMERO D...,NOME DO MUNICÍPIO UF,REGISTRO FOTOGRÁFICO JOANOPOLIS AGROPECUARIA L...,FORD/CARGO 1415 NÚMERO DO EQUIPAMENTO OU INSTR...,notificacaoPenalidade_BSC4C89_1262001AA6014095...,a:\Multas\Placas\BSC4C89\a_vencer\notificacaoP...,NOTIFICAÇÃO DE PENALIDADE\nIDENTIFICAÇÃO DA PE...,NaN


In [9]:
# ===== CÉLULA 5: RESUMO POR PLACA =====
resumo = (
    df_multas
    .groupby(["placa", "status"])
    .agg(qtd=("arquivo", "count"), valor_total=("valor_num", "sum"))
    .reset_index()
    .pivot(index="placa", columns="status", values=["qtd", "valor_total"])
    .fillna(0)
)

resumo

placa


In [ ]:
# ===== CÉLULA 6: EXPORTAÇÃO =====
saida = BASE_DIR / "resumo_multas.xlsx"
with pd.ExcelWriter(saida, engine="openpyxl") as writer:
    df_multas.drop(columns=["texto"]).to_excel(
        writer, sheet_name="Detalhado", index=False)
    resumo.to_excel(writer, sheet_name="Resumo por Placa")
print(f"Exportado: {saida}")

ValueError: zip() argument 2 is longer than argument 1